# R5 · Suites — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/r5_suites/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/r5_suites/03_notebook.ipynb`, créée par `python tools/start_chapter.py R5`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | R5.1 à R5.8, R5.11, R5.12 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | R5.14 | Terme d'une suite arithmétique et somme géométrique · `arithmetic_term`, `geometric_sum` | 🔨 | ★ | 15 |
|  | R5.15 | Itérer une récurrence · `iterate` | 🔨 | ★ | 15 |
|  | R5.16 | Moyennes mobile et exponentielle · `moving_average`, `exponential_moving_average` | 🔨 | ★★ | 25 |
| B | R5.17 | Les mêmes calculs avec NumPy : `np.cumsum`, `np.cumprod`, `np.convolve` | 📦 | ★ | 15 |
|  | R5.18 | Prédire : vers quoi tend u(n + 1) = 0,75 u(n) + 2 ? | 🔮 | ★ | 10 |
|  | R5.19 | Lisser un signal bruité : l'effet de la fenêtre et de β | 🔬 | ★★ | 20 |
|  | R5.20 | Des suites qui convergent, divergent, oscillent | 🎨 | ★ | 15 |
| C | R5.21 | Chasse à l'erreur : un décalage d'indice dans une somme | 🐛 | ★ | 15 |
|  | R5.22 | Une boucle ou une formule ? Mesurer le temps de calcul | 🛠️ | ★ | 15 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="R5")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : termes, raisons, sommes, sommes infinies, intérêts composés, moyennes mobiles et exponentielles.
- Écrire les cinq fonctions de `mymath/sequences.py` et les tester.
- Refaire ces calculs avec NumPy, prédire le comportement d'une récurrence, régler un lissage, tracer des suites, chasser un décalage d'indice et chronométrer une formule.

**Rappel express.** Arithmétique : $u_n = u_0 + n\,r$ ; géométrique : $u_n = u_0\,q^n$. Somme arithmétique : nombre de termes × (premier + dernier) / 2 ; somme géométrique de $n$ termes : premier × $\frac{1 - q^n}{1 - q}$ ($q \ne 1$). Si $\lvert q \rvert < 1$ : $q^n \to 0$ et la somme infinie vaut premier ÷ $(1 - q)$. Moyenne exponentielle : $v_0 = 0$, $v_t = \beta\,v_{t-1} + (1 - \beta)\,x_t$.

## Partie 0 · Vérifier tes exercices papier (✏️ R5.1 à R5.8, 🔗 R5.11 et R5.12)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un nombre avec un point décimal, `-1.5`, arrondi comme le demande la question ;
- une fraction exacte entre guillemets, `"5/7"` ;
- une expression en `n` entre guillemets, `"3*n**2 + 1"` (`*` pour multiplier, `**` pour la puissance) ;
- une liste de nombres entre crochets, dans l'ordre demandé ;
- vrai ou faux par `True` ou `False` ; un numéro de réponse proposée par un entier.

Les réponses pas encore remplies affichent ⏳.

**Ex R5.1 — Calculer des termes : explicite et récurrence**

In [ ]:
# Ex R5.1: the values you found on paper
answer_R5_1a = ...  # a) an integer
answer_R5_1b = ...  # b) an integer
answer_R5_1c = ...  # c) a list of 3 numbers
answer_R5_1d = ...  # d) an integer
answer_R5_1e = ...  # e) an expression in n, as text
answer_R5_1f = ...  # f) an expression in n, as text
answer_R5_1g = ...  # g) an integer

for letter, answer in zip("abcdefg", [answer_R5_1a, answer_R5_1b, answer_R5_1c, answer_R5_1d, answer_R5_1e, answer_R5_1f, answer_R5_1g]):
    wbm.check(f"R5.1{letter}", answer)

**Ex R5.2 — Terme général d'une suite arithmétique**

In [ ]:
# Ex R5.2: the values you found on paper
answer_R5_2a = ...  # a) an expression in n, as text
answer_R5_2b = ...  # b) an integer
answer_R5_2c = ...  # c) a number
answer_R5_2d = ...  # d) an integer
answer_R5_2e = ...  # e) an integer
answer_R5_2f = ...  # f) True or False
answer_R5_2g = ...  # g) an integer

for letter, answer in zip("abcdefg", [answer_R5_2a, answer_R5_2b, answer_R5_2c, answer_R5_2d, answer_R5_2e, answer_R5_2f, answer_R5_2g]):
    wbm.check(f"R5.2{letter}", answer)

**Ex R5.3 — Terme général d'une suite géométrique**

In [ ]:
# Ex R5.3: the values you found on paper
answer_R5_3a = ...  # a) an expression in n, as text
answer_R5_3b = ...  # b) an integer
answer_R5_3c = ...  # c) an integer
answer_R5_3d = ...  # d) a fraction, as text
answer_R5_3e = ...  # e) an integer
answer_R5_3f = ...  # f) an integer
answer_R5_3g = ...  # g) an integer

for letter, answer in zip("abcdefg", [answer_R5_3a, answer_R5_3b, answer_R5_3c, answer_R5_3d, answer_R5_3e, answer_R5_3f, answer_R5_3g]):
    wbm.check(f"R5.3{letter}", answer)

**Ex R5.4 — Sommes arithmétiques et géométriques**

In [ ]:
# Ex R5.4: the values you found on paper
answer_R5_4a = ...  # a) an integer
answer_R5_4b = ...  # b) an integer
answer_R5_4c = ...  # c) a number
answer_R5_4d = ...  # d) an integer
answer_R5_4e = ...  # e) an integer
answer_R5_4f = ...  # f) an integer
answer_R5_4g = ...  # g) an integer
answer_R5_4h = ...  # h) an expression in n, as text

for letter, answer in zip("abcdefgh", [answer_R5_4a, answer_R5_4b, answer_R5_4c, answer_R5_4d, answer_R5_4e, answer_R5_4f, answer_R5_4g, answer_R5_4h]):
    wbm.check(f"R5.4{letter}", answer)

**Ex R5.5 — Sommes infinies d'une suite géométrique**

In [ ]:
# Ex R5.5: the values you found on paper
answer_R5_5a = ...  # a) an integer
answer_R5_5b = ...  # b) a fraction, as text
answer_R5_5c = ...  # c) a fraction, as text
answer_R5_5d = ...  # d) a fraction, as text
answer_R5_5e = ...  # e) True or False
answer_R5_5f = ...  # f) an integer

for letter, answer in zip("abcdef", [answer_R5_5a, answer_R5_5b, answer_R5_5c, answer_R5_5d, answer_R5_5e, answer_R5_5f]):
    wbm.check(f"R5.5{letter}", answer)

**Ex R5.6 — Intérêts composés et actualisation**

In [ ]:
# Ex R5.6: the values you found on paper
answer_R5_6a = ...  # a) a number
answer_R5_6b = ...  # b) a number (2 decimals)
answer_R5_6c = ...  # c) a number
answer_R5_6d = ...  # d) an integer
answer_R5_6e = ...  # e) a number (2 decimals)
answer_R5_6f = ...  # f) a number (2 decimals)
answer_R5_6g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_R5_6a, answer_R5_6b, answer_R5_6c, answer_R5_6d, answer_R5_6e, answer_R5_6f, answer_R5_6g]):
    wbm.check(f"R5.6{letter}", answer)

**Ex R5.7 — Une moyenne exponentielle à la main sur cinq valeurs**

In [ ]:
# Ex R5.7: the values you found on paper
answer_R5_7a = ...  # a) a list of 5 numbers (4 decimals)
answer_R5_7b = ...  # b) a list of 3 numbers (2 decimals)
answer_R5_7c = ...  # c) a number
answer_R5_7d = ...  # d) True or False
answer_R5_7e = ...  # e) a number
answer_R5_7f = ...  # f) a number

for letter, answer in zip("abcdef", [answer_R5_7a, answer_R5_7b, answer_R5_7c, answer_R5_7d, answer_R5_7e, answer_R5_7f]):
    wbm.check(f"R5.7{letter}", answer)

**Ex R5.8 — Retrouver le premier terme et la raison à partir de deux termes**

In [ ]:
# Ex R5.8: the values you found on paper
answer_R5_8a = ...  # a) an integer
answer_R5_8b = ...  # b) an integer
answer_R5_8c = ...  # c) an integer
answer_R5_8d = ...  # d) an integer
answer_R5_8e = ...  # e) a number
answer_R5_8f = ...  # f) a number
answer_R5_8g = ...  # g) a list (6 decimals)

for letter, answer in zip("abcdefg", [answer_R5_8a, answer_R5_8b, answer_R5_8c, answer_R5_8d, answer_R5_8e, answer_R5_8f, answer_R5_8g]):
    wbm.check(f"R5.8{letter}", answer)

**Ex R5.11 — Actualiser : ce que vaut aujourd'hui une somme versée chaque année**

In [ ]:
# Ex R5.11: the values you found on paper
answer_R5_11a = ...  # a) a number (2 decimals)
answer_R5_11b = ...  # b) a number (4 decimals)
answer_R5_11c = ...  # c) a number (2 decimals)
answer_R5_11d = ...  # d) 1 or 2
answer_R5_11e = ...  # e) a number (2 decimals)

for letter, answer in zip("abcde", [answer_R5_11a, answer_R5_11b, answer_R5_11c, answer_R5_11d, answer_R5_11e]):
    wbm.check(f"R5.11{letter}", answer)

**Ex R5.12 — Une moyenne mobile qui glisse : la mettre à jour sans tout recalculer**

In [ ]:
# Ex R5.12: the values you found on paper
answer_R5_12a = ...  # a) a number
answer_R5_12b = ...  # b) a number
answer_R5_12c = ...  # c) a number (1 to 4)
answer_R5_12d = ...  # d) a number
answer_R5_12e = ...  # e) a list of 2 integers
answer_R5_12f = ...  # f) a number

for letter, answer in zip("abcdef", [answer_R5_12a, answer_R5_12b, answer_R5_12c, answer_R5_12d, answer_R5_12e, answer_R5_12f]):
    wbm.check(f"R5.12{letter}", answer)

## Partie A · Ta librairie : termes, sommes, récurrences et lissages

Fiche §1, §2, §4 et §6. Les cinq fonctions de `mymath/sequences.py`.

In [ ]:
# Tools of the notebook (parts A to C)
import math
import time

import matplotlib.pyplot as plt
import numpy as np

from wbm.nbtools import checked, drawn, run_pytest_group


def raised(function, *args):
    """The name of the exception that function(*args) raises, or "aucune" if it returns.

    NotImplementedError (a TODO not done yet) goes through, for wbm.attempt to show ⏳."""
    try:
        function(*args)
    except NotImplementedError:
        raise
    except Exception as exc:  # noqa: BLE001 - any other error is the answer
        return type(exc).__name__
    return "aucune"

### Ex R5.14 — Terme d'une suite arithmétique et somme géométrique · `arithmetic_term`, `geometric_sum` 🔨 ★ ⏱️ 15 min
**Objectif :** programmer le terme général d'une suite arithmétique et la somme d'une suite géométrique, sans boucle.  
**Prérequis :** fiche §2 et §4 · **mymath :** `sequences.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme dans les chapitres précédents). Ouvre ta copie `mon_travail/mymath/sequences.py`, lis la docstring de la fonction en entier, remplace la ligne `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. Les tests comparent tes résultats à des calculs exacts (`fractions`, SymPy), à NumPy et à SciPy.

Dans `mon_travail/mymath/sequences.py` :
- `arithmetic_term(u0, r, n)` : le terme de rang `n` de la suite arithmétique de premier terme `u0` et de raison `r` (le rang 0 est `u0`), avec la formule explicite, sans boucle ;
- `geometric_sum(u0, q, n)` : la somme des `n` premiers termes `u0 + u0*q + … + u0*q**(n - 1)`, avec la formule de la fiche §4, sans boucle ; **`n` est le nombre de termes** (la somme de 0 terme vaut 0) ; pense au cas `q == 1`, où la formule divise par zéro.

La vérification calcule deux termes ($u_0 = -4$, $r = 7$, rang 12 ; $u_0 = 2{,}5$, $r = -0{,}5$, rang 9), la somme de 6 termes de premier terme 3 et de raison 1,5, deux cas particuliers ($q = 1$ ; $n = 0$), puis une somme de 40 termes de raison $\frac{1}{4}$.

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="R5")  # reload your saved file

In [ ]:
with checked("R5.14a"):
    wbm.check("R5.14a", [mymath.sequences.arithmetic_term(-4, 7, 12), mymath.sequences.arithmetic_term(2.5, -0.5, 9)],
              computed=True)
with checked("R5.14b"):
    wbm.check("R5.14b", mymath.sequences.geometric_sum(3, 1.5, 6), computed=True)
with checked("R5.14c"):
    wbm.check("R5.14c", [mymath.sequences.geometric_sum(5, 1, 7), mymath.sequences.geometric_sum(8, 3, 0)],
              computed=True)
with checked("R5.14d"):
    wbm.check("R5.14d", mymath.sequences.geometric_sum(6, 0.25, 40), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_r5_sequences.py", "test_arithmetic_term_ or test_geometric_sum_")

### Ex R5.15 — Itérer une récurrence · `iterate` 🔨 ★ ⏱️ 15 min
**Objectif :** programmer le calcul des premiers termes d'une suite définie par récurrence, pour n'importe quelle règle.  
**Prérequis :** fiche §1 ; workbookIA 0A (`lambda`, fonctions passées en argument) · **mymath :** `sequences.py` · **Parcours :** C, E

Écris `iterate(f, u0, n)` dans `mon_travail/mymath/sequences.py` : la liste `[u(0), u(1), …, u(n)]` des termes de la suite définie par `u(0) = u0` et `u(k + 1) = f(u(k))`. **`n` est le dernier rang calculé** : la liste a `n + 1` termes. `f` est une fonction, souvent une `lambda` : `iterate(lambda u: 3 * u - 1, 2, 3)` calcule les termes de rang 0 à 3 de $u_{k+1} = 3u_k - 1$ partie de 2.

La vérification calcule :
- a) les termes de rang 0 à 4 de $u_{k+1} = 10 - \frac{u_k}{2}$, partie de 0 ;
- b) le nombre de termes de `iterate(abs, -3, 10)` ;
- c) `iterate(abs, -3, 0)` ;
- d) le terme de rang 10 de $u_{k+1} = 1 + \frac{1}{u_k}$, partie de 1.

Puis, dans ta copie : si la suite de d) converge vers une limite $\ell$, quelle équation $\ell$ vérifie-t-elle (fiche §5) ? Résous-la (R3, second degré) et compare à d).

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="R5")  # reload your saved file

In [ ]:
with checked("R5.15"):
    iterate = mymath.sequences.iterate
    wbm.check("R5.15a", iterate(lambda u: 10 - u / 2, 0, 4), computed=True)
    wbm.check("R5.15b", len(iterate(abs, -3, 10)), computed=True)
    wbm.check("R5.15c", iterate(abs, -3, 0), computed=True)
    wbm.check("R5.15d", iterate(lambda u: 1 + 1 / u, 1, 10)[-1], computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_r5_sequences.py", "test_iterate_")

### Ex R5.16 — Moyennes mobile et exponentielle · `moving_average`, `exponential_moving_average` 🔨 ★★ ⏱️ 25 min
**Objectif :** programmer les deux lissages de la fiche, avec leurs cas limites et leurs erreurs.  
**Prérequis :** fiche §6 ; R5.7 et R5.12 · **mymath :** `sequences.py` · **Parcours :** C, E

Dans `mon_travail/mymath/sequences.py` :
- `moving_average(values, window)` : la liste des `len(values) - window + 1` moyennes de `window` valeurs consécutives (la moyenne de `values[0:window]`, puis de `values[1:window + 1]`, etc.) ; une `ValueError` si `window` n'est pas compris entre 1 et `len(values)` ;
- `exponential_moving_average(values, beta)` : la liste `[v(1), …, v(T)]` de la moyenne exponentielle $v_0 = 0$, $v_t = \beta\,v_{t-1} + (1 - \beta)\,x_t$ ; une `ValueError` si `beta` n'est pas dans $[0 \;;\; 1[$.

La vérification calcule :
- a) les moyennes mobiles sur 3 valeurs de `[4, 9, 6, 10, 2, 12]` ;
- b) le nombre de moyennes mobiles sur 7 valeurs d'une liste de 50 valeurs ;
- c) le nom de l'erreur levée pour une fenêtre de 3 sur 2 valeurs (avec `raised`, de la cellule des outils), puis d) pour une fenêtre de 0 ;
- e) la moyenne exponentielle de `[40, 0, 20, 20]` avec $\beta = 0{,}75$ ;
- f) la moyenne exponentielle de `[3, 1, 4]` avec $\beta = 0$ ;
- g) le nom de l'erreur levée pour $\beta = 1$.

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="R5")  # reload your saved file

In [ ]:
with checked("R5.16a"):
    wbm.check("R5.16a", mymath.sequences.moving_average([4, 9, 6, 10, 2, 12], 3), computed=True)
with checked("R5.16b"):
    wbm.check("R5.16b", len(mymath.sequences.moving_average(list(range(50)), 7)), computed=True)
with checked("R5.16c"):
    wbm.check("R5.16c", raised(mymath.sequences.moving_average, [1, 2], 3), computed=True)
with checked("R5.16d"):
    wbm.check("R5.16d", raised(mymath.sequences.moving_average, [1, 2], 0), computed=True)
with checked("R5.16e"):
    wbm.check("R5.16e", mymath.sequences.exponential_moving_average([40, 0, 20, 20], 0.75), computed=True)
with checked("R5.16f"):
    wbm.check("R5.16f", mymath.sequences.exponential_moving_average([3, 1, 4], 0), computed=True)
with checked("R5.16g"):
    wbm.check("R5.16g", raised(mymath.sequences.exponential_moving_average, [1], 1), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_r5_sequences.py", "test_moving_average_ or test_exponential_moving_average_")

## Partie B · NumPy, une prédiction et des expériences

Fiche §4 à §6. Les mêmes calculs en une ligne avec NumPy, une récurrence à prédire, le réglage d'un lissage, puis des suites qui convergent ou non, en image.

### Ex R5.17 — Les mêmes calculs avec NumPy : `np.cumsum`, `np.cumprod`, `np.convolve` 📦 ★ ⏱️ 15 min
**Objectif :** calculer des sommes partielles, des produits cumulés et des moyennes mobiles en une ligne avec NumPy.  
**Prérequis :** fiche §4 et §6 ; workbookIA 0A §100.8.1 à §100.8.5 (arrays, calcul vectorisé, réductions) · **Parcours :** C

Trois fonctions de NumPy font en une ligne ce que tu as programmé avec des boucles :
- `np.cumsum(a)` rend les **sommes partielles** de `a` : `np.cumsum([2, 7, 1])` vaut `[2, 9, 10]` ;
- `np.cumprod(a)` rend les **produits cumulés** : `np.cumprod([2, 7, 1])` vaut `[2, 14, 14]` ;
- `np.convolve(a, w, mode="valid")` fait glisser la liste de poids `w` le long de `a` et rend, à chaque position où `w` tient en entier, la somme des produits : avec `w = np.ones(k) / k`, c'est la moyenne mobile sur `k` valeurs.

Et `3 + 2 * np.arange(5)` calcule d'un coup les termes $3 + 2k$ pour $k = 0$ à 4 (calcul vectorisé, 0A §100.8.3). Calcule, chacun **en une ligne** :
- a) les 10 premières sommes partielles de la suite arithmétique $u_k = 4 + 3k$ (de $u_0$ à $u_9$) ;
- b) les 6 premières sommes partielles de la suite géométrique $5 \times 1{,}2^k$ ;
- c) le capital à la fin de chacune des quatre années d'un placement de 1 000 €, dont le taux change chaque année : +2 %, +5 %, −3 %, +4 % (les taux sont dans `rates`) ;
- d) la moyenne mobile sur 2 valeurs de `x`.

Puis, dans ta copie : pourquoi la suite des capitaux de c) n'est-elle pas géométrique ?

In [ ]:
rates = np.array([0.02, 0.05, -0.03, 0.04])
x = np.array([5, 7, 3, 9, 11, 1, 8])

answer_R5_17a = ...  # a) an array of 10 partial sums
answer_R5_17b = ...  # b) an array of 6 partial sums
answer_R5_17c = ...  # c) an array of 4 capitals
answer_R5_17d = ...  # d) an array of moving averages

In [ ]:
for letter, answer in zip("abcd", [answer_R5_17a, answer_R5_17b, answer_R5_17c, answer_R5_17d]):
    wbm.check(f"R5.17{letter}", answer)

### Ex R5.18 — Prédire : vers quoi tend u(n + 1) = 0,75 u(n) + 2 ? 🔮 ★ ⏱️ 10 min
**Objectif :** prédire le comportement d'une récurrence affine avec le point fixe et l'écart géométrique, puis le confronter à l'expérience.  
**Prérequis :** fiche §1 et §5 · **Parcours :** C, E

On étudie la suite définie par $u_{n+1} = 0{,}75\,u_n + 2$, avec deux premiers termes possibles : $u_0 = 0$ ou $u_0 = 100$. **Avant d'exécuter quoi que ce soit**, prédis avec la fiche §5, sans calculer de termes :
- a) la limite de la suite partie de $u_0 = 0$ ;
- b) la limite de la suite partie de $u_0 = 100$ ;
- c) partie de $u_0 = 0$, combien de pas faut-il pour que $u_n$ soit à moins de 0,01 de sa limite ? 1 : moins de 10 ; 2 : entre 10 et 50 ; 3 : plus de 50.

Écris tes trois prédictions, vérifie-les, puis lance l'expérience.

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_R5_18a = ...  # a) a number, written BEFORE running the experiment below
prediction_R5_18b = ...  # b) a number
prediction_R5_18c = ...  # c) 1, 2 or 3

In [ ]:
for letter, prediction in zip("abc", [prediction_R5_18a, prediction_R5_18b, prediction_R5_18c]):
    wbm.check(f"R5.18{letter}", prediction)

**L'expérience.** Elle ne s'exécute qu'une fois tes trois prédictions écrites.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_R5_18a, prediction_R5_18b, prediction_R5_18c]):
    print("⏳ Écris d'abord tes trois prédictions, puis exécute à nouveau.")
else:
    def run(u0, steps):
        """[u(0), …, u(steps)] for u(n + 1) = 0.75 u(n) + 2."""
        terms = [float(u0)]
        for _ in range(steps):
            terms.append(0.75 * terms[-1] + 2)
        return terms


    from_0, from_100 = run(0, 40), run(100, 40)
    far = run(0, 500)[-1]                                    # the value after 500 steps: where the sequence settles
    print("depuis 0   :", [round(u, 3) for u in from_0[:6]], "…", round(from_0[-1], 6))
    print("depuis 100 :", [round(u, 3) for u in from_100[:6]], "…", round(from_100[-1], 6))
    print("valeur après 500 pas :", far)
    print("premier rang à moins de 0,01 de cette valeur, depuis 0 :",
          next(n for n, u in enumerate(from_0) if abs(u - far) < 0.01))
    fig, ax = plt.subplots(figsize=(7, 3.6))
    ax.plot(from_0, "o", markersize=3, label="u₀ = 0")
    ax.plot(from_100, "o", markersize=3, label="u₀ = 100")
    ax.axhline(far, color="gray", linestyle=":", label="valeur après 500 pas")
    ax.set_xlabel("n")
    ax.set_ylabel("uₙ")
    ax.set_title("u(n + 1) = 0,75 u(n) + 2, depuis deux premiers termes")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()

**Après l'expérience.** d) Le premier rang $n$ pour lequel $u_n$ est à moins de 0,01 de sa limite, partie de $u_0 = 0$ (lis-le dans la sortie). Puis, dans ta copie : écris $u_n$ en fonction de $n$ pour $u_0 = 0$ (fiche §5), et compare les deux suites.

In [ ]:
answer_R5_18d = ...  # d) an integer

In [ ]:
wbm.check("R5.18d", answer_R5_18d)

### Ex R5.19 — Lisser un signal bruité : l'effet de la fenêtre et de β 🔬 ★★ ⏱️ 20 min
**Objectif :** mesurer le compromis d'un lissage : trop peu, il reste du bruit ; trop, la courbe est en retard.  
**Prérequis :** fiche §6 ; R5.16, R5.17 · **Parcours :** C

La cellule ci-dessous fabrique un signal : une **vraie tendance** (une sinusoïde, qu'on ne connaît pas en pratique) à laquelle s'ajoute du **bruit**. Elle donne trois outils : `moving(x, k)` (la moyenne mobile, alignée sur le temps : sa valeur à l'instant $t$ utilise $x_{t-k+1}, \dots, x_t$), `ema(x, beta)` (la moyenne exponentielle) et `error(smoothed)`, l'écart moyen (la racine de la moyenne des carrés des écarts) entre une courbe lissée et la vraie tendance, à partir de $t = 60$.

Calcule :
- a) `window_errors`, la liste des erreurs `error(moving(x, k))` pour chaque fenêtre de `windows` (dans cet ordre) ;
- b) `beta_errors`, la liste des erreurs `error(ema(x, beta))` pour chaque valeur de `betas`.

Puis, en lisant tes listes et la figure :
- c) la fenêtre de `windows` qui donne la plus petite erreur ;
- d) la valeur de `betas` qui donne la plus petite erreur ;
- e) avec une fenêtre de 61, la courbe lissée est-elle en avance (1) ou en retard (2) sur la vraie tendance ?

Dans ta copie : pourquoi l'erreur diminue-t-elle d'abord, puis augmente-t-elle, quand la fenêtre grandit ?

In [ ]:
from scipy.signal import lfilter

rng = np.random.default_rng(2026)
t = np.arange(300)
trend = 10 + 4 * np.sin(2 * np.pi * t / 120)           # the true signal, unknown in practice
x = trend + rng.normal(0, 1.5, size=t.size)             # what we measure: the signal plus noise


def moving(x, k):
    """Moving average over k values, aligned on time: entry t is the mean of x[t-k+1..t] (NaN for t < k - 1)."""
    out = np.full(len(x), np.nan)
    out[k - 1:] = np.convolve(x, np.ones(k) / k, mode="valid")
    return out


def ema(x, beta):
    """Exponential moving average v(t) = beta v(t-1) + (1 - beta) x(t), v(0) = 0 (one line of SciPy)."""
    return lfilter([1 - beta], [1, -beta], x)


def error(smoothed, start=60):
    """Root mean square gap between a smoothed curve and the true trend, from time `start` on."""
    return float(np.sqrt(np.mean((smoothed[start:] - trend[start:]) ** 2)))


windows = [1, 3, 9, 25, 61]
betas = [0, 0.5, 0.75, 0.9, 0.99]

In [ ]:
window_errors = ...  # a) a list: error(moving(x, k)) for each k of windows
beta_errors = ...    # b) a list: error(ema(x, beta)) for each beta of betas
answer_R5_19c = ...  # c) a window of windows
answer_R5_19d = ...  # d) a value of betas
answer_R5_19e = ...  # e) 1 (ahead) or 2 (behind)

In [ ]:
with checked("R5.19"):
    wbm.check("R5.19a", window_errors, computed=True)
    wbm.check("R5.19b", beta_errors, computed=True)
for letter, answer in zip("cde", [answer_R5_19c, answer_R5_19d, answer_R5_19e]):
    wbm.check(f"R5.19{letter}", answer)

**La figure** : les erreurs en fonction de la fenêtre et de β, puis deux lissages.

In [ ]:
with checked("R5.19 figure"):
    if window_errors is ... or beta_errors is ...:
        print("⏳ Calcule d'abord window_errors et beta_errors.")
    else:
        fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
        axes[0].plot(windows, window_errors, "o-")
        axes[0].set_xlabel("fenêtre k")
        axes[0].set_title("Erreur de la moyenne mobile")
        axes[1].plot(betas, beta_errors, "o-", color="C1")
        axes[1].set_xlabel("β")
        axes[1].set_title("Erreur de la moyenne exponentielle")
        axes[2].plot(t, x, ".", color="gray", markersize=2, label="mesures")
        axes[2].plot(t, trend, color="black", linewidth=1, label="vraie tendance")
        axes[2].plot(t, moving(x, 9), color="C0", label="fenêtre 9")
        axes[2].plot(t, moving(x, 61), color="C3", label="fenêtre 61")
        axes[2].set_xlabel("t")
        axes[2].set_title("Deux fenêtres")
        axes[2].legend(fontsize=8)
        for ax in axes[:2]:
            ax.set_ylabel("erreur")
            ax.grid(True, alpha=0.3)
        fig.tight_layout()
        plt.show()

### Ex R5.20 — Des suites qui convergent, divergent, oscillent 🎨 ★ ⏱️ 15 min
**Objectif :** voir les comportements d'une suite (converger, osciller, diverger), et reconnaître une limite sur un graphique.  
**Prérequis :** fiche §5 ; R0 (matplotlib) · **Parcours :** C

Complète `plot_sequences(axes)`, qui reçoit **quatre axes** côte à côte et trace, pour $n = 0$ à 30, les termes de :
- (a) $5 - 4 \times 0{,}7^n$ dans `axes[0]` ;
- (b) $2 + 3 \times (-0{,}75)^n$ dans `axes[1]` ;
- (c) $(-1)^n \times \frac{n}{n + 1}$ dans `axes[2]` ;
- (d) $\sqrt{n}$ dans `axes[3]`.

Trace des **points** (une suite n'est définie qu'aux entiers) : `ax.plot(ns, values, "o")`, avec `ns = np.arange(31)` et le calcul vectorisé de NumPy ; un titre par graphique. La cellule de vérification crée les axes, puis la figure s'affiche. Puis :
- a) les lettres des suites qui convergent, dans l'ordre alphabétique (un texte, comme `"bd"`) ;
- b) la liste de leurs limites, dans le même ordre ;
- c) la suite (c) se rapproche tour à tour de deux nombres : la liste de ces deux nombres.

Dans ta copie : pour chaque suite, une phrase qui décrit son comportement (converge en restant du même côté, converge en oscillant, oscille sans limite, tend vers $+\infty$).

In [ ]:
def plot_sequences(axes):
    """Draw the terms of (a), (b), (c), (d) for n = 0 to 30 as points, one sequence in each of the four axes."""
    raise NotImplementedError("trace les quatre suites dans axes[0] à axes[3]")


answer_R5_20a = ...  # a) letters, as text
answer_R5_20b = ...  # b) a list of limits
answer_R5_20c = ...  # c) a list of two numbers

In [ ]:
with checked("R5.20 figure"):
    fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
    try:
        created = drawn(plot_sequences, axes)
    except BaseException:
        plt.close(fig)                                     # no empty figure while plot_sequences is not written yet
        raise
    if created:
        plt.close(fig)
        print("❌ Ex R5.20 : ta fonction ouvre sa propre figure (plt.figure ou plt.subplots) : dessine dans les axes "
              "qu'elle reçoit, avec axes[0].plot(...), …")
    else:
        fig.tight_layout()
        plt.show()
for letter, answer in zip("abc", [answer_R5_20a, answer_R5_20b, answer_R5_20c]):
    wbm.check(f"R5.20{letter}", answer)

**Vérifications automatiques** (la figure elle-même se juge à l'œil).

In [ ]:
with checked("R5.20 vérifications"):
    fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
    try:
        created = drawn(plot_sequences, axes)
    finally:
        plt.close(fig)                                     # only the checks below are shown here
    rules = [lambda n: 5 - 4 * 0.7**n, lambda n: 2 + 3 * (-0.75) ** n, lambda n: (-1) ** n * n / (n + 1),
             lambda n: math.sqrt(n)]
    problems = []
    if created:
        problems.append("ta fonction ouvre sa propre figure : dessine dans les axes reçus (axes[0].plot, …)")
    else:
        for k, (name, ax, rule) in enumerate(zip("abcd", axes, rules)):
            points = []
            for line in ax.lines:
                points += list(zip(np.asarray(line.get_xdata(), dtype=float), np.asarray(line.get_ydata(), dtype=float)))
            for collection in ax.collections:
                points += [tuple(p) for p in np.asarray(collection.get_offsets(), dtype=float)]
            at_integers = {n: [py for px, py in points if abs(px - n) < 1e-9] for n in range(21)}
            if not points:
                problems.append(f"rien n'est tracé dans axes[{k}] : dessine ({name}) avec axes[{k}].plot(...), pas plt.plot")
                continue
            wrong = [n for n, ys in at_integers.items() if ys and not any(abs(y - rule(n)) < 1e-6 for y in ys)]
            missing = [n for n, ys in at_integers.items() if not ys]
            if wrong:
                problems.append(f"les termes de ({name}) dans axes[{k}] ne sont pas les bons (dès le rang {wrong[0]})")
            elif missing:
                problems.append(f"il manque des termes de ({name}) dans axes[{k}] (au moins celui de rang {missing[0]})")
    if problems:
        print("❌ Ex R5.20 : à revoir : " + " ; ".join(problems) + ".")
    else:
        print("✅ Ex R5.20 : les quatre suites sont tracées, chacune dans son graphique.")

## Partie C · Indices et temps de calcul

Fiche §4. Deux décalages d'indice à trouver, puis une boucle et une formule chronométrées.

### Ex R5.21 — Chasse à l'erreur : un décalage d'indice dans une somme 🐛 ★ ⏱️ 15 min
**Objectif :** repérer un décalage d'indice dans une boucle et dans une formule de somme, puis le corriger.  
**Prérequis :** fiche §4 ; R4 §5 (bornes d'un range) · **Parcours :** C, E

Une épargnante dépose `first` euros la première année, puis augmente son dépôt de `rate` (2 % s'écrit `0.02`) chaque année : ses dépôts sont `first`, `first*(1 + rate)`, `first*(1 + rate)**2`, … Les deux fonctions ci-dessous doivent calculer le total de ses `years` premiers dépôts, l'une avec une boucle, l'autre avec la formule de la fiche §4. Elles contiennent chacune une erreur d'indice. **Sans rien exécuter d'abord**, trouve-les.

```python
def total_savings(first, rate, years):
    """The total of `years` yearly deposits that grow by `rate` each year: first, first*(1 + rate), …"""
    q = 1 + rate                                        # (A)
    total = 0                                           # (B)
    for k in range(1, years):                           # (C)
        total += first * q**k                           # (D)
    return total                                        # (E)


def total_formula(first, rate, years):
    """The same total, with the formula of the sum of a geometric sequence."""
    q = 1 + rate                                        # (F)
    return first * (1 - q ** (years - 1)) / (1 - q)    # (G)
```

- a) les lettres des lignes fausses, dans l'ordre alphabétique (un texte) ;
- b) combien de dépôts la boucle fautive additionne-t-elle pour `years = 10` ?

Puis corrige les deux fonctions dans la cellule ci-dessous (elle contient le code fautif) : la vérification calcule c), la liste `[total_savings(1000, 0.02, 5), total_formula(1000, 0.02, 5)]`, qui doit donner deux fois le même total.

In [ ]:
answer_R5_21a = ...  # a) letters, as text
answer_R5_21b = ...  # b) an integer


# correct the two functions below
def total_savings(first, rate, years):
    """The total of `years` yearly deposits that grow by `rate` each year: first, first*(1 + rate), …"""
    q = 1 + rate                                        # (A)
    total = 0                                           # (B)
    for k in range(1, years):                           # (C)
        total += first * q**k                           # (D)
    return total                                        # (E)


def total_formula(first, rate, years):
    """The same total, with the formula of the sum of a geometric sequence."""
    q = 1 + rate                                        # (F)
    return first * (1 - q ** (years - 1)) / (1 - q)    # (G)

In [ ]:
for letter, answer in zip("ab", [answer_R5_21a, answer_R5_21b]):
    wbm.check(f"R5.21{letter}", answer)
if answer_R5_21a is ...:
    print("⏳ Ex R5.21c : trouve d'abord les lignes fausses (a), puis corrige les deux fonctions.")
else:
    with checked("R5.21c"):
        wbm.check("R5.21c", [total_savings(1000, 0.02, 5), total_formula(1000, 0.02, 5)], computed=True)

### Ex R5.22 — Une boucle ou une formule ? Mesurer le temps de calcul 🛠️ ★ ⏱️ 15 min
**Objectif :** mesurer le temps d'un calcul avec `time.perf_counter`, et voir ce qu'une formule fait gagner sur une boucle.  
**Prérequis :** fiche §4 ; R5.14 · **Parcours :** C

`time.perf_counter()` rend un temps en secondes, très précis : la durée d'un calcul est la différence entre la valeur lue juste après et la valeur lue juste avant. Complète la cellule : pour $n = 10^3$, $10^4$, $10^5$ et $10^6$, et $q = 0{,}999\,999$, mesure le temps de `loop_sum(q, n)` et celui de `formula_sum(q, n)`, et affiche $n$, les deux temps et l'écart entre les deux résultats.

Puis, dans ta copie :
1. Quand $n$ est multiplié par 10, par combien le temps de la boucle est-il (à peu près) multiplié ? Et celui de la formule ?
2. Les deux résultats sont-ils exactement égaux ? D'où vient l'écart (R1 : les flottants) ?
3. Pourquoi les temps changent-ils un peu d'une exécution à l'autre ?

In [ ]:
def loop_sum(q, n):
    """1 + q + … + q**(n - 1), added term by term."""
    total = 0.0
    for k in range(n):
        total += q**k
    return total


def formula_sum(q, n):
    """The same sum with the formula (q != 1)."""
    return (1 - q**n) / (1 - q)


# For n = 10**3, 10**4, 10**5 and 10**6, with q = 0.999999: measure the time of each function with
# time.perf_counter() (before and after the call), and print n, the two times and the gap between the two results.

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Calcules-tu les termes d'une suite explicite ou par récurrence, et reconnais-tu une suite arithmétique ou géométrique ?
2. Calcules-tu une somme de termes, et une somme infinie géométrique, en comptant bien les termes ?
3. Sais-tu dire si une suite converge, et calculer une moyenne mobile et une moyenne exponentielle ?

**La suite** : R6, *Exponentielle et logarithmes*. Dans workbookIA, continue le chapitre 0A au rythme de ton tableau de bord.